# Supervised Learning: Linear Models
### Interactive Notebook for AI/ML Interview Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score, accuracy_score
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Linear Regression from Scratch

In [ ]:
# Generate data: y = 3x + 7 + noise
X = np.random.uniform(0, 10, 100)
y = 3 * X + 7 + np.random.normal(0, 2, 100)

# Gradient descent
w, b = 0.0, 0.0
lr = 0.001
losses = []
for epoch in range(200):
    y_pred = w * X + b
    loss = np.mean((y_pred - y)**2)
    losses.append(loss)
    dw = 2 * np.mean((y_pred - y) * X)
    db = 2 * np.mean(y_pred - y)
    w -= lr * dw
    b -= lr * db

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X, y, s=10, alpha=0.5)
axes[0].plot(sorted(X), [w*x+b for x in sorted(X)], 'r-', linewidth=2)
axes[0].set_title(f'Linear Fit: y = {w:.2f}x + {b:.2f}')
axes[1].plot(losses); axes[1].set_title('Loss Curve'); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

---
## 2. Logistic Regression — Decision Boundary

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_informative=2, random_state=42, n_clusters_per_class=1)
clf = LogisticRegression().fit(X, y)

# Plot decision boundary
x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
plt.scatter(X[:,0], X[:,1], c=y, cmap='RdBu', edgecolors='black', s=30)
plt.title(f'Logistic Regression (Accuracy: {clf.score(X,y):.1%})')
plt.tight_layout(); plt.show()

---
## 3. Ridge vs Lasso Regularization

In [ ]:
# Coefficient paths as alpha varies
np.random.seed(42)
X_reg = np.random.randn(100, 10)
true_coef = np.array([3, -2, 0, 0, 1.5, 0, 0, -1, 0, 0])  # sparse
y_reg = X_reg @ true_coef + np.random.normal(0, 0.5, 100)

alphas = np.logspace(-2, 3, 50)
ridge_coefs = [Ridge(alpha=a).fit(X_reg, y_reg).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=10000).fit(X_reg, y_reg).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for coefs, ax, name in zip([ridge_coefs, lasso_coefs], axes, ['Ridge (L2)', 'Lasso (L1)']):
    coefs = np.array(coefs)
    for i in range(10):
        ax.plot(alphas, coefs[:, i], label=f'w{i}')
    ax.set_xscale('log'); ax.set_xlabel('Alpha'); ax.set_ylabel('Coefficient')
    ax.set_title(f'{name} — Coefficient Paths')
    ax.axhline(y=0, color='black', linestyle=':')
axes[0].legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()
print('Lasso drives coefficients to exactly zero (feature selection!)')
print('Ridge shrinks but never zeros out coefficients')

---
## 4. Polynomial Regression — Overfitting Demo

In [ ]:
X_poly = np.sort(np.random.uniform(0, 1, 20))
y_poly = np.sin(2 * np.pi * X_poly) + np.random.normal(0, 0.3, 20)
X_test = np.linspace(0, 1, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, degree, label in zip(axes, [1, 4, 15], ['Underfit', 'Good Fit', 'Overfit']):
    poly = PolynomialFeatures(degree)
    X_p = poly.fit_transform(X_poly.reshape(-1,1))
    lr = LinearRegression().fit(X_p, y_poly)
    y_pred = lr.predict(poly.transform(X_test.reshape(-1,1)))
    ax.scatter(X_poly, y_poly, s=30, zorder=5)
    ax.plot(X_test, y_pred, 'r-', linewidth=2)
    ax.set_title(f'{label} (degree={degree})')
    ax.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

---
## Key Interview Takeaways

1. **Linear regression** — closed-form (normal equation) or gradient descent; understand both
2. **Logistic regression** — outputs probabilities via sigmoid; decision boundary is linear
3. **Ridge (L2)** — shrinks coefficients; good for multicollinearity
4. **Lasso (L1)** — zeros out coefficients; built-in feature selection
5. **Polynomial regression** — demonstrates bias-variance tradeoff directly